# The classifier zoo: decision boundaries

**Classification** predicts a discrete label for an example:

1. **binary**: two classes, typically *normal* vs *abnormal* (spam / ham, benign / malicious);
2. **multi-class**: one of several exclusive classes (malware *family*);
3. **multi-label**: several labels may hold at once (a file that is both a *dropper* and a *keylogger*).

Every classifier is a way of drawing a **decision boundary** in feature space. Different families draw different boundaries, and
that shape decides both *what the model can learn* and *how easily an attacker can push a point across it*.

We use eight families on three 2-D datasets: a *linearly separable* one, a *round* one (no straight line works), and noisy *moons*.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.datasets import make_moons
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

plt.rcParams["figure.dpi"] = 100


def models():
    return {
        "Logistic regression": LogisticRegression(),
        "Naive Bayes": GaussianNB(),
        "SVM (RBF)": SVC(),
        "Neural network": MLPClassifier(hidden_layer_sizes=(32, 32), max_iter=5000, random_state=0),
        "1-NN": KNeighborsClassifier(n_neighbors=1),
        "Decision tree": DecisionTreeClassifier(random_state=0),
        "Random forest": RandomForestClassifier(random_state=0),
        "Gradient boosting": GradientBoostingClassifier(random_state=0),
        "Voting (LR + kNN + RF)": VotingClassifier(
            [
                ("lr", LogisticRegression()),
                ("knn", KNeighborsClassifier(n_neighbors=5)),
                ("rf", RandomForestClassifier(random_state=0)),
            ],
            voting="soft",
        ),
    }


def probability(clf, points):
    """P(class 1); for the SVM (no probabilities) a sigmoid of the signed distance to the margin."""
    if hasattr(clf, "predict_proba"):
        return clf.predict_proba(points)[:, 1]
    return 1 / (1 + np.exp(-clf.decision_function(points)))


def boundary_grid(X_tr, y_tr, X_te, y_te, title):
    lo, hi = X_tr.min(axis=0) - 0.7, X_tr.max(axis=0) + 0.7
    xx, yy = np.meshgrid(np.linspace(lo[0], hi[0], 200), np.linspace(lo[1], hi[1], 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    fig, axes = plt.subplots(3, 3, figsize=(11, 10.5))
    for ax, (name, clf) in zip(axes.ravel(), models().items()):
        clf.fit(X_tr, y_tr)
        ax.contourf(
            xx, yy, probability(clf, grid).reshape(xx.shape), levels=25, cmap="RdBu", vmin=0, vmax=1, alpha=0.85
        )
        ax.scatter(*X_tr.T, c=y_tr, cmap="RdBu", vmin=-0.2, vmax=1.2, edgecolor="white", s=35)
        if len(X_te):
            ax.scatter(*X_te.T, c=y_te, cmap="RdBu", vmin=-0.2, vmax=1.2, edgecolor="k", marker="s", s=25)
            ax.set_title(f"{name}\ntrain {clf.score(X_tr, y_tr):.2f}  test {clf.score(X_te, y_te):.2f}", fontsize=9)
        else:
            ax.set_title(f"{name}\ntrain {clf.score(X_tr, y_tr):.2f}", fontsize=9)
        ax.set(xticks=[], yticks=[])
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## 1. A linearly separable problem

36 points on a grid (`datasets/toy_dataset_00.csv`); all points are used for training, since the point is the *shape* of the boundary.

In [ ]:
df = pd.read_csv("../../datasets/toy_dataset_00.csv")
Xa, ya = df[["X1", "X2"]].to_numpy(dtype=float), df["Y"].to_numpy()
boundary_grid(Xa, ya, Xa[:0], ya[:0], "linear toy dataset")

## 2. A round problem

A straight line cannot separate an inner disc from a ring. Logistic regression and Naive Bayes have the wrong *inductive bias*;
the non-linear models adapt. This is **bias** in the sense of the bias-variance trade-off (`notebook_09`).

In [ ]:
df = pd.read_csv("../../datasets/toy_dataset_02.csv")
Xb, yb = df[["X1", "X2"]].to_numpy(dtype=float), df["Y"].to_numpy()
boundary_grid(Xb, yb, Xb[:0], yb[:0], "round toy dataset")

## 3. Noisy data: flexible models memorise the noise

300 noisy "moons", 70/30 split (squares are *test* points). Compare train and test accuracy in the titles: 1-NN, the tree
and the random forest reach 1.00 on the training set, and that is exactly the *overfitting* symptom.

In [ ]:
Xm, ym = make_moons(n_samples=300, noise=0.3, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(Xm, ym, test_size=0.3, stratify=ym, random_state=1)
boundary_grid(X_tr, y_tr, X_te, y_te, "noisy moons")

## 4. How far is a point from the boundary?

For the attacker the relevant number is the **distance to the boundary**: the smallest perturbation that changes the prediction.
We estimate it *without any gradient* (black-box): for each test point we try many random directions and, for each, find how far we must
walk before the label flips; the distance is the minimum over directions.

In [ ]:
def distance_to_boundary(clf, X, n_dirs=64, steps=np.linspace(0.02, 2.0, 40), seed=0):
    rng = np.random.default_rng(seed)
    base = clf.predict(X)
    dirs = rng.normal(size=(n_dirs, X.shape[1]))
    dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)
    P = X[:, None, None, :] + steps[None, None, :, None] * dirs[None, :, None, :]  # (n, dirs, steps, 2)
    flipped = clf.predict(P.reshape(-1, X.shape[1])).reshape(len(X), n_dirs, len(steps)) != base[:, None, None]
    first = np.where(
        flipped.any(axis=2), flipped.argmax(axis=2), len(steps) - 1
    )  # first step that flips (else the maximum)
    return steps[first].min(axis=1)


print(f"{'model':26}{'test acc':>9}{'median distance to boundary':>30}")
results = {}
for name, clf in models().items():
    if name.startswith("Voting"):
        continue
    clf.fit(X_tr, y_tr)
    d = distance_to_boundary(clf, X_te)
    results[name] = d
    print(f"{name:26}{clf.score(X_te, y_te):9.2f}{np.median(d):30.3f}")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.boxplot(results.values(), tick_labels=results.keys(), orientation="horizontal")
ax.set(xlabel="distance from a test point to the decision boundary")
plt.tight_layout()
plt.show()

Accuracy and robustness are **different properties**: two models with the same test accuracy can put their boundaries at very
different distances from the data. Models whose boundary snakes between the training points (1-NN, tree, forest) leave many points
*close* to it. Points close to the boundary are the ones an attacker can push across with a small change (`notebook_11`).

## Exercises

1. Increase `noise` to 0.5. Which models lose the most test accuracy?
2. Set `n_neighbors=15` for the k-NN and `max_depth=3` for the tree. What happens to the train/test gap and to the distance to the boundary?
3. Add a `LinearSVC` and compare its boundary with logistic regression. Are they the same? Why (not)?
4. **Ensembles.** Is the voting classifier's boundary smoother than its members'? Does its distance to the boundary increase?